# cr-train Colab Quickstart

이 노트북은 Colab 같은 새 환경에서 `git clone -> uv sync -> HF_TOKEN 설정 -> quick start 학습`까지 한 번에 따라가는 용도입니다.

- 대상 데이터셋: `Hermanni/sen12mscr`
- 권장 환경: Colab GPU 런타임
- 기본 데모: `official` split, 짧은 학습/검증/테스트 budget


## 1. uv 설치, 저장소 클론, 의존성 설치

`uv`는 노트북 실행기를 대체하는 게 아니라, 의존성 설치를 빠르고 일관되게 처리하는 역할입니다.

여기서는 다음 두 가지를 같이 처리합니다.

- 현재 노트북 커널에 locked dependency 설치
- `git clone`한 `src/` layout 패키지를 바로 import할 수 있게 `sys.path` 연결


In [ ]:
%pip install -q uv

import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/cr-train")
requirements_path = Path("/tmp/cr-train-requirements.txt")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/smturtle2/cr-train.git", str(repo_dir)],
        check=True,
    )

os.chdir(repo_dir)
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--format",
        "requirements.txt",
        "--no-hashes",
        "--no-emit-project",
        "-o",
        str(requirements_path),
    ],
    check=True,
)
subprocess.run(
    [
        "uv",
        "pip",
        "sync",
        "--system",
        "--break-system-packages",
        str(requirements_path),
    ],
    check=True,
)

src_dir = repo_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"working directory: {repo_dir}")
print(f"src path added: {src_dir}")


## 2. Hugging Face 토큰 설정

토큰이 없어도 public access로 동작할 수는 있지만, rate limit이 더 빨리 걸릴 수 있습니다.

Colab Secrets에 `HF_TOKEN`을 넣어두는 것을 우선 권장합니다. 필요하면 바로 아래 **셀 입력칸**에 직접 붙여넣어도 됩니다.

In [ ]:
# @title 2. Hugging Face 토큰 설정
HF_TOKEN_INPUT = ""  # @param {type:"string"}

import os

token = os.environ.get("HF_TOKEN", "") or HF_TOKEN_INPUT.strip()

if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN configured.")
else:
    print("HF_TOKEN not found. Public access still works, but rate limits may be tighter.")
    print("Paste the token into the form field above or configure Colab Secrets, then rerun this cell.")


## 3. 공개 API import와 device 확인

In [ ]:
import torch
from torch import nn

from cr_train import MAE, Trainer, TrainerConfig, build_sen12mscr_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


## 4. DataLoader 생성

기본 split은 `official`이고, train 경로는 항상 `reshard() -> shuffle(...)` 순서를 지킵니다.

In [ ]:
train_loader, val_loader, test_loader = build_sen12mscr_loaders(
    batch_size=4,
    seed=7,
    split="official",
    shuffle_buffer_size=16,
)

print("loaders ready")


## 5. 배치 확인

기본 전처리:

- `cloudy`, `target`: `clip(0, 10000) / 10000.0`
- `sar`: `clip(-25, 0)` 후 `(x + 25) / 25`


In [ ]:
batch = next(iter(train_loader))

print("sar", batch["inputs"]["sar"].shape, batch["inputs"]["sar"].dtype)
print("cloudy", batch["inputs"]["cloudy"].shape, batch["inputs"]["cloudy"].dtype)
print("target", batch["target"].shape, batch["target"].dtype)
print("sar range", float(batch["inputs"]["sar"].min()), float(batch["inputs"]["sar"].max()))
print(
    "cloudy range",
    float(batch["inputs"]["cloudy"].min()),
    float(batch["inputs"]["cloudy"].max()),
)
print("metadata keys", list(batch["metadata"].keys()))


## 6. 모델 정의

In [ ]:
class TinyCloudRemovalNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(15, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 13, kernel_size=1),
        )

    def forward(self, sar: torch.Tensor, cloudy: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([sar, cloudy], dim=1))


## 7. 학습 설정

In [ ]:
model = TinyCloudRemovalNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    metrics=[MAE()],
    config=TrainerConfig(
        max_epochs=1,
        train_max_batches=4,
        val_max_batches=2,
        test_max_batches=2,
        checkpoint_dir="artifacts/colab-notebook",
    ),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)


## 8. 학습 실행

In [ ]:
for history in trainer.step():
    print(history)


## 9. 테스트 평가

In [ ]:
print(trainer.test())
